In [1]:
from langchain_chroma import Chroma, vectorstores
from langchain_openai import OpenAIEmbeddings

vectorstores = Chroma(
    collection_name="resume_docs",
    embedding_function=OpenAIEmbeddings( # 기존 벡터스토어를 로드할 때 embedding_function 파라미터를 반드시 지정해야 함
        base_url="http://localhost:1234/v1",
        model="text-embedding-nomic-embed-text-v2-moe",
        check_embedding_ctx_length=False
    ),
    persist_directory="./chroma_db"
)

print(f"로드된 문서 수: {vectorstores._collection.count()}")

로드된 문서 수: 9


# 유사도 검색

In [3]:
results = vectorstores.similarity_search("대학교", k=3)

for i, doc in enumerate(results):
    print(f"\n--- 결과 {i+1} ---")
    print(f"내용: {doc.page_content[:150]}...")
    print(f"메타데이터: {doc.metadata}")


--- 결과 1 ---
내용: 이 민 호 남 만  24 세  24  0727 000994
 DB Inc  & DB FIS  2024 년   채 용연 계 형   인 턴 사 원   모 집
국적대한 민 국 생년월 일 2000 02 08
한문 이 름 李 旻 淏 영문 이 름 LEE MINHO 이 메일 min...
메타데이터: {'title': '내가 작성한 이력서', 'creationdate': '2024-08-18T12:20:21+00:00', 'total_pages': 4, 'source': './data/DB_fis-내가 작성한 이력서.pdf', 'moddate': '2024-08-18T12:20:21+00:00', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36 Edg/127.0.0.0', 'page': 0, 'producer': 'Skia/PDF m127', 'page_label': '1'}

--- 결과 2 ---
내용:  
컴 퓨 터 활 용 능력
 
수상경 력
교 육 이 수 사 항
그 렙
리 눅 스   시스 템   및   커 널   전문 가   과정
이 수 기 간 2023 10 02   2024 03 02 교 육 시 간 600 시 간
주요내용 리 눅 스   시스 템   프 로 그 래 ...
메타데이터: {'moddate': '2024-08-18T12:20:21+00:00', 'title': '내가 작성한 이력서', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36 Edg/127.0.0.0', 'page_label': '3', 'source': './data/DB_fis-내가 작성한 이력서.pdf', 'producer

## 유사도 점수와 함께 접수

In [4]:
results_with_scores = vectorstores.similarity_search_with_score(
    "대학교", k=3
)

for doc, score in results_with_scores:
    print(f"점수: {score:.4f} | {doc.page_content[:80]}...")

점수: 1.3670 | 을   수   있을   것 이 라   기 대합니 다    입 사   초 기 에는   프 로 젝 트의   다 양 한   역할 을   경 험 하 며...
점수: 1.3830 | 이 민 호 남 만  24 세  24  0727 000994
 DB Inc  & DB FIS  2024 년   채 용연 계 형   인 턴 사 원 ...
점수: 1.4096 |   대
학 력 사 항   추 가
 논 문   첨 부
경 력 사 항
 
포트 폴 리 오   첨 부
직 장경 력 재직   회 사   수 1개
티 앤...


# Retriever 패턴
LangGraph 워크플로우에서 벡터스토어를 사용할 때는 as_retriever() 메서드로 Retriever 인터페이스를 활용할 수 있습니다.

In [6]:
# 기본 Retriever
retriever = vectorstores.as_retriever(search_kwargs={"k": 5})

# Retriever로 검색
docs = retriever.invoke("이 이력서를 쓴 사람의 이름")
print(f"검색된 문서 수: {len(docs)}")

# MMR (Maximal Marginal Relevance) — 다양성 확보
mmr_retriever = vectorstores.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 10},
)
docs = mmr_retriever.invoke("이 이력서를 쓴 사람의 이름")
print(f"검색된 문서 수(MMR): {len(docs)}")

검색된 문서 수: 5
검색된 문서 수(MMR): 5
